In [1]:
import mp_api
import pandas as pd
from mp_offline.client import MPOffline, MaterialSummary
from pymatgen.core.composition import Composition
from pymatgen.core import Structure
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
from collections import Counter

from copy import deepcopy

Steps

1. Prepare the data
2. Select materials that are binary/ternary
3. Form groups based on composition
4. Form groups based on local geometry
5. Filter by volume changes and band gaps
6. Form the final candidates for calculation

## Loading data using `mp_offline`

In [4]:
client = MPOffline()

col =['material_id', 'energy_above_hull', 'composition', 'structure', 'json_data']
entries = client.query_all(MaterialSummary.energy_above_hull < 0.01, 
                           project=col)

df = pd.DataFrame(entries, columns=col)
df['composition'] = df['composition'].apply(Composition)
df['structure'] = df['structure'].apply(Structure.from_dict)
df['band_gap'] = df['json_data'].apply(lambda x: x['band_gap'])

df['nelems'] = df['composition'].apply(lambda x: len(x.keys()))

df['reduced_composition'] = df.composition.apply(lambda x: x.get_reduced_composition_and_factor()[0])

df.set_index('material_id', inplace=True)

df_t = df.loc[(df['nelems'] >= 2)
            & (df['band_gap'] < 0.1), :]

0it [00:00, ?it/s]

/home/bonan/miniconda3/envs/work/lib/python3.12/site-packages/pymatgen/core/periodic_table.py:289: UserWarning: No Pauling electronegativity for Ne. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(
/home/bonan/miniconda3/envs/work/lib/python3.12/site-packages/pymatgen/core/periodic_table.py:289: UserWarning: No Pauling electronegativity for Ar. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(
/home/bonan/miniconda3/envs/work/lib/python3.12/site-packages/pymatgen/core/periodic_table.py:289: UserWarning: No Pauling electronegativity for He. Setting to NaN. This has no physical meaning, and is mainly done to avoid errors caused by the code expecting a float.
  warnings.warn(


In [5]:
df_t

,energy_above_hull,composition,structure,json_data,band_gap,nelems,reduced_composition
material_id,,,,,,,
mp-571576,0.000000,"(La, Te)","[[0.08373719 4.60778935 2.53527043] La, [6.884...","{'_id': {'$oid': '654ac03b3e1d94017b73506e'}, ...",0.0,2,"(La, Te)"
mp-1217429,0.000000,"(Te, Rh)","[[5.94527447 1.94074118 6.73414968] Te, [0. ...","{'_id': {'$oid': '654abd8f3e1d94017b5ce9ab'}, ...",0.0,2,"(Te, Rh)"
mp-1216708,0.000000,"(V, Ni)","[[1.12590589 2.84934605 6.05435926] V, [1.1259...","{'_id': {'$oid': '654abed33e1d94017b678f9b'}, ...",0.0,2,"(V, Ni)"
mp-569073,0.000000,"(Li, Sn)","[[1.56926692 3.60932262 4.30354952] Li, [ 1.56...","{'_id': {'$oid': '654ac33c3e1d94017b81abff'}, ...",0.0,2,"(Li, Sn)"
mp-569147,0.000000,"(U, P)","[[0.72791066 3.66196818 2.22873142] U, [3.6099...","{'_id': {'$oid': '654abc553e1d94017b52a3f8'}, ...",0.0,2,"(U, P)"
...,...,...,...,...,...,...,...
mp-21402,0.009980,"(Pr, Cr, Si, C)","[[0. 0. 0.] Pr, [0. 2.007927 2.703088] C...","{'_id': {'$oid': '654ac1813e1d94017b798bea'}, ...",0.0,4,"(Pr, Cr, Si, C)"
mp-1183915,0.009983,"(Eu, Cd, Hg)","[[5.84785244 5.84785244 5.84785244] Eu, [1.949...","{'_id': {'$oid': '654ac03e3e1d94017b73752d'}, ...",0.0,3,"(Eu, Cd, Hg)"
mp-1078458,0.009989,"(Cr, Fe, O)","[[ 2.52304694 -1.45647242 4.39516619] Cr, [ 2...","{'_id': {'$oid': '654abd973e1d94017b5d42c1'}, ...",0.0,3,"(Cr, Fe, O)"
